# XR Client Cursors

Examples of responding to generic user controller interaction.

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: xr client cursors")
imd_runner.load(0)

In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

## Line drawing

Defining a mode where pressing and holding the primary button on any controller draws red lines in the simulation space:

<video controls src="./assets/line_drawing.webm">

In [3]:
from nanover.jupyter import Mode
from nanover.jupyter.utilities import make_id_generator


make_new_line_id = make_id_generator()
# mapping of line id to line points
LINE_POINTS = {}
# mapping of cursor to currently drawn line if any
CURSOR_CURRENT_LINE_ID = {}


class DrawLineMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        # when the primary button is pressed on a cursor, assign a new line to this cursor
        if button == "primary":
            line_id = make_new_line_id()
            LINE_POINTS[line_id] = []
            CURSOR_CURRENT_LINE_ID[key] = line_id

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        # when the primary button is released, unassign any line from this cursor
        if button == "primary":
            CURSOR_CURRENT_LINE_ID.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        # if there is a line assigned to this cursor, do drawing
        line_id = CURSOR_CURRENT_LINE_ID.get(key, None)
        if line_id is None:
            return
        # convert the cursor's scene space position to a simulation space position
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        # add the simulation space position to the assigned line
        LINE_POINTS[line_id].append(next_pos)
        # update the line visuals to reflect the added point
        utilities.objects.update_line(f"line.{line_id}", positions=LINE_POINTS[line_id], size=0.05, color=[1.0, 0.0, 0.0, 1.0])


utilities.use_interaction_modes()
utilities.add_interaction_mode(DrawLineMode, "draw line", icon="✏️")

## Object dragging

Defining a mode where pressing and holding the primary button on any controller grabs an object and moves it around:

<video controls src="./assets/object_dragging.webm">

In [4]:
import numpy as np
from nanover.jupyter import Mode


OBJECT_RADIUS = 1
OBJECT_COLOR = [1.0, 0.0, 0.0, 1.0]
HOVER_COLOR = [1.0, 1.0, 1.0, 0.5]
CURSOR_RADIUS = 0

# mapping of cursor to currently grabbed object if any
CURSOR_GRABBED_OBJECT_ID = {}
# mapping of object id to object position
OBJECT_POSITIONS = {}


def redraw_object(object_id):
    utilities.objects.update_shape(object_id, position=OBJECT_POSITIONS[object_id], color=OBJECT_COLOR, size=OBJECT_RADIUS)


def intersect_objects(position):
    for object_id, object_pos in OBJECT_POSITIONS.items():
        if np.linalg.norm(np.subtract(position, object_pos)) < OBJECT_RADIUS + CURSOR_RADIUS:
            return object_id
    return None


# put some objects in the scene
for i in range(4):
    OBJECT_POSITIONS[str(i)] = [i, i, i]
    redraw_object(str(i))


class MoveObjectMode(Mode):
    def on_button_pressed(self, *, key: str, cursor: dict, button: str):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])
        hovered = intersect_objects(next_pos)
        available = hovered not in CURSOR_GRABBED_OBJECT_ID.values()

        # grab hovered object if not already grabbed
        if button == "primary" and available:
            CURSOR_GRABBED_OBJECT_ID[key] = hovered

    def on_button_released(self, *, key: str, cursor: dict, button: str):
        # release grabbed
        if button == "primary":
            CURSOR_GRABBED_OBJECT_ID.pop(key, None)

    def on_cursor_updated(self, *, key: str, cursor: dict):
        next_pos = utilities.scene_transform.point_parent_to_local(cursor["position"])

        # if this cursor has grabbed something, update its position
        if key in CURSOR_GRABBED_OBJECT_ID:
            object_id = CURSOR_GRABBED_OBJECT_ID[key]
            OBJECT_POSITIONS[object_id] = next_pos
            redraw_object(object_id)

        # show/remove hover graphic is this cursor hovers something
        hovered = intersect_objects(next_pos)
        if hovered is None:
            utilities.objects.update_shape(f"hovered.{key}")
        else:
            utilities.objects.update_shape(f"hovered.{key}", position=OBJECT_POSITIONS[hovered], color=HOVER_COLOR, size=OBJECT_RADIUS * 1.1)


utilities.use_interaction_modes()
utilities.add_interaction_mode(MoveObjectMode, "move object", icon="✊")